# English to Spanish Neural Machine Translation

This notebook demonstrates how to build and train a neural machine translation model to translate English sentences to Spanish. It covers both GRU-based Recurrent Neural Networks (RNNs) and Transformer models.

In [ ]:
!wget http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip
!unzip -q spa-eng.zip

--2026-08-21 10:50:21--  http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.195.207, 172.253.117.207, 142.250.99.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.195.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2638744 (2.5M) [application/zip]
Saving to: ‘spa-eng.zip’

spa-eng.zip         100%[===================>]   2.52M  --.-KB/s    in 0.007s  

2026-08-21 10:50:21 (370 MB/s) - ‘spa-eng.zip’ saved [2638744/2638744]



## 1. Data Loading and Preprocessing

In [ ]:
text_file = "spa-eng/spa.txt"
with open(text_file) as f:
  lines = f.read().split("\n")[:-1]
text_pairs = []
for line in lines:
  english, spanish = line.split("\t")
  spanish = "[start] " + spanish + " [end]"
  text_pairs.append((english, spanish))

In [ ]:
import random
print(random.choice(text_pairs))

('How do I fix this problem?', '[start] ¿Cómo resuelvo este problema? [end]')


In [ ]:
#splitting the data into train, validation and test sets
import random
random.shuffle(text_pairs)
num_val_samples = int(0.15 * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples
train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples:num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples:]

In [ ]:
import tensorflow as tf
import string
import re
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras import preprocessing
import numpy as np

strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

#custom text vectorization for espanol which preserves '[' and ']'
def custom_standardization(input_string):
  lowercase = tf.strings.lower(input_string)
  return tf.strings.regex_replace(lowercase, f"[{re.escape(strip_chars)}]", "")

vocab_size = 15000
sequence_length = 20

#English Layer
source_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,)

#Spanish layer
target_vectorization = layers.TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,)

train_english_texts = [pair[0] for pair in train_pairs]
train_spanish_texts = [pair[1] for pair in train_pairs]
source_vectorization.adapt(train_english_texts)
target_vectorization.adapt(train_spanish_texts)

In [ ]:
#preparing the dataset for the translation task
batch_size = 64

def format_dataset(eng, spa):
  eng = source_vectorization(eng)
  spa = target_vectorization(spa)
  return ({
      "english": eng,
      "spanish": spa[:, :-1],
  }, spa[:, 1:])

def make_dataset(pairs):
  eng_texts, spa_texts = zip(*pairs)
  eng_texts = list(eng_texts)
  spa_texts = list(spa_texts) # Corrected typo: spa_text to spa_texts
  dataset = tf.data.Dataset.from_tensor_slices((eng_texts, spa_texts))
  dataset = dataset.batch(batch_size)
  dataset = dataset.map(format_dataset, num_parallel_calls=4)
  return dataset.shuffle(2048).prefetch(16).cache()

train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

In [ ]:
for inputs, targets in train_ds.take(1):
   print(f"inputs['english'].shape: {inputs['english'].shape}")
   print(f"inputs['spanish'].shape: {inputs['spanish'].shape}")
   print(f"targets.shape: {targets.shape}")


inputs['english'].shape: (64, 20)
inputs['spanish'].shape: (64, 20)
targets.shape: (64, 20)


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
#GRU Based encoder
embed_dim = 256
latent_dim = 1024

source = keras.Input(shape=(None,), dtype="int64", name="english")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(source)
encoded_source = layers.Bidirectional(
    layers.GRU(latent_dim), merge_mode="sum")(x)

#GRU based decoder and end to end model
past_target = keras.Input(shape=(None,), dtype="int64", name="spanish")
x = layers.Embedding(vocab_size, embed_dim, mask_zero=True)(past_target)
decoder_gru = layers.GRU(latent_dim, return_sequences=True)
x = decoder_gru(x, initial_state=encoded_source)
x = layers.Dropout(0.5)(x)
target_next_step = layers.Dense(vocab_size, activation="softmax")(x)
seq2seq_rnn = keras.Model([source, past_target], target_next_step)

## 2. GRU-based Sequence-to-Sequence Model

In [ ]:
seq2seq_rnn.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

seq2seq_rnn.fit(train_ds, epochs=15, validation_data=val_ds)

Epoch 1/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 254s 187ms/step - accuracy: 0.3188 - loss: 4.6781 - val_accuracy: 0.3816 - val_loss: 3.9505
Epoch 2/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 243s 187ms/step - accuracy: 0.4136 - loss: 3.7369 - val_accuracy: 0.4670 - val_loss: 3.2627
Epoch 3/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 246s 189ms/step - accuracy: 0.4705 - loss: 3.2278 - val_accuracy: 0.5142 - val_loss: 2.8927
Epoch 4/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 242s 186ms/step - accuracy: 0.5112 - loss: 2.8748 - val_accuracy: 0.5495 - val_loss: 2.6390
Epoch 5/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 242s 186ms/step - accuracy: 0.5444 - loss: 2.6008 - val_accuracy: 0.5771 - val_loss: 2.4601
Epoch 6/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 242s 186ms/step - accuracy: 0.5728 - loss: 2.3786 - val_accuracy: 0.5948 - val_loss: 2.3347
Epoch 7/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 242s 186ms/step - accuracy: 0.5965 - loss: 2.1943 - val_accuracy: 0.6122 - val_loss: 2.2282
Epoch 8/15
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 243s 186ms/step - ac

### 2.1 Model Training

In [ ]:
#translating new sentences with our RNN encoder and decoder

import numpy as np
spa_vocab = target_vectorization.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))
max_decoded_sentence_length = 20

def decode_sequence(input_sentence):
  tokenized_input_sentence = source_vectorization([input_sentence])
  decoded_sentence = "[start]"
  for i in range(max_decoded_sentence_length):
    tokenized_target_sentence = target_vectorization([decoded_sentence])
    next_token_predictions = seq2seq_rnn.predict(
        [tokenized_input_sentence, tokenized_target_sentence])
    sampled_token_index = np.argmax(next_token_predictions[0, i, :])
    sampled_token = spa_index_lookup[sampled_token_index]
    decoded_sentence += " " + sampled_token
    if sampled_token == "[end]":
      break
  return decoded_sentence

### 2.2 Inference with GRU Model

In [ ]:
test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(20):
  input_sentence = random.choice(test_eng_texts)
  print("-")
  print(input_sentence)
  print(decode_sequence(input_sentence))

-
I'm sure Tom will be able to win.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 329ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
[start] estoy seguro de que tom podrá ganar [end]
-
Tom wants to buy a present for Mary.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
[start] tom quiere comprar un regalo de mary [end]
-
I think he is a good driver.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms

## 3. Transformer Model

In [ ]:
class TransformerEncoder(keras.Layer):
    def __init__(self, hidden_dim, intermediate_dim, num_heads):
        super().__init__()
        key_dim = hidden_dim // num_heads
        self.self_attention = layers.MultiHeadAttention(num_heads, key_dim)
        self.self_attention_layernorm = layers.LayerNormalization()
        self.feed_forward_1 = layers.Dense(intermediate_dim, activation="relu")
        self.feed_forward_2 = layers.Dense(hidden_dim)
        self.feed_forward_layernorm = layers.LayerNormalization()

    def call(self, source, source_mask):
        residual = x = source
        mask = source_mask[:, None, :]
        x = self.self_attention(query=x, key=x, value=x, attention_mask=mask)
        x = x + residual
        x = self.self_attention_layernorm(x)
        residual = x
        x = self.feed_forward_1(x)
        x = self.feed_forward_2(x)
        x = x + residual
        x = self.feed_forward_layernorm(x)
        return x

In [ ]:
class TransformerDecoder(layers.Layer):
  def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
    super().__init__(**kwargs)
    self.embed_dim = embed_dim
    self.dense_dim = dense_dim
    self.num_heads = num_heads
    self.attention_1 = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_dim)
    self.attention_2 = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_dim)
    self.dense_proj = keras.Sequential(
        [layers.Dense(dense_dim, activation="relu"),
         layers.Dense(embed_dim),])
    self.layernorm_1 = layers.LayerNormalization()
    self.layernorm_2 = layers.LayerNormalization()
    self.layernorm_3 = layers.LayerNormalization()
    self.support_masking = True

  def get_config(self):
    config = super().get_config()
    config.update({
        "embed_dim": self.embed_dim,
        "num_heads": self.num_heads,
        "dense_dim": self.dense_dim,})
    return config

  def get_causal_attention_mask(self, inputs):
    input_shape = tf.shape(inputs)
    batch_size, sequence_length = input_shape[0], input_shape[1]
    i = tf.range(sequence_length)[:, tf.newaxis]
    j = tf.range(sequence_length)
    mask = tf.cast(i >= j, dtype="int32")
    mask = tf.reshape(mask, (1, input_shape[1], input_shape[1]))
    mult = tf.concat(
        [tf.expand_dims(batch_size, -1),
         tf.constant([1, 1], dtype=tf.int32)], axis=0)
    return tf.tile(mask, mult)

  def call(self, inputs, encoder_outputs, mask=None):
    causal_mask = self.get_causal_attention_mask(inputs)
    if mask is not None:
      padding_mask = tf.cast(
          mask[:, tf.newaxis, :], dtype="int32")
      padding_mask = tf.minimum(padding_mask, causal_mask)
    else:
      padding_mask = causal_mask # Ensure padding_mask is defined even if mask is None

    attention_output_1 = self.attention_1( # Corrected from self.attention
        query=inputs,
        value=inputs,
        key=inputs,
        attention_mask=causal_mask)
    attention_output_1 = self.layernorm_1(inputs + attention_output_1)
    attention_output_2 = self.attention_2( # Corrected from self.attention
        query=attention_output_1,
        value=encoder_outputs,
        key=encoder_outputs,
        attention_mask=padding_mask,)
    attention_output_2 = self.layernorm_2(
        attention_output_1 + attention_output_2)
    proj_output = self.dense_proj(attention_output_2)
    return self.layernorm_3(attention_output_2 + proj_output)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras import ops

class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim)
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=embed_dim)
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        # Corrected: Using keras.ops.not_equal instead of tf.math.not_equal
        return keras.ops.not_equal(inputs, 0) # Assuming 0 is padding token

    def get_config(self):
        config = super().get_config()
        config.update({
            "sequence_length": self.sequence_length,
            "vocab_size": self.vocab_size,
            "embed_dim": self.embed_dim,
        })
        return config

#end to end transformer
embed_dim = 256
dense_dim = 2048
num_heads = 8

encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="english")
embedded_encoder_inputs = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(encoder_inputs)
encoder_mask = embedded_encoder_inputs._keras_mask # Get the mask from the embedding layer
encoder_outputs = TransformerEncoder(embed_dim, dense_dim, num_heads)(embedded_encoder_inputs, encoder_mask)

decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="spanish")
embedded_decoder_inputs = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(decoder_inputs)
decoder_mask = embedded_decoder_inputs._keras_mask # Get the mask from the embedding layer
x = TransformerDecoder(embed_dim, dense_dim, num_heads)(embedded_decoder_inputs, encoder_outputs, decoder_mask)
x = layers.Dropout(0.5)(x)
decoder_outputs = layers.Dense(vocab_size, activation="softmax")(x)
transformer = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

transformer.compile(
    optimizer="rmsprop",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

transformer.fit(train_ds, epochs=30, validation_data=val_ds)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'transformer_encoder_2' (of type TransformerEncoder) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Epoch 1/30


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'transformer_decoder_1' (of type TransformerDecoder) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


1302/1302 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.7347 - loss: 2.0745

/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'transformer_encoder_2' (of type TransformerEncoder) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'transformer_decoder_1' (of type TransformerDecoder) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


1302/1302 ━━━━━━━━━━━━━━━━━━━━ 133s 86ms/step - accuracy: 0.7664 - loss: 1.6075 - val_accuracy: 0.8062 - val_loss: 1.1910
Epoch 2/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 93s 71ms/step - accuracy: 0.8135 - loss: 1.1465 - val_accuracy: 0.8324 - val_loss: 0.9987
Epoch 3/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 92s 71ms/step - accuracy: 0.8337 - loss: 0.9977 - val_accuracy: 0.8440 - val_loss: 0.9201
Epoch 4/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 91s 70ms/step - accuracy: 0.8460 - loss: 0.9129 - val_accuracy: 0.8499 - val_loss: 0.8930
Epoch 5/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 92s 70ms/step - accuracy: 0.8552 - loss: 0.8550 - val_accuracy: 0.8572 - val_loss: 0.8514
Epoch 6/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 91s 70ms/step - accuracy: 0.8622 - loss: 0.8109 - val_accuracy: 0.8590 - val_loss: 0.8499
Epoch 7/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 91s 70ms/step - accuracy: 0.8685 - loss: 0.7750 - val_accuracy: 0.8621 - val_loss: 0.8391
Epoch 8/30
1302/1302 ━━━━━━━━━━━━━━━━━━━━ 91s 70ms/step - accuracy: 0.8746 - loss: 0.7

### 3.1 Transformer Model Training

In [ ]:
import numpy as np
spa_vocab = target_vectorization.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))
max_decoded_sentence_length = 20

def decode_sequence(input_sentence):
  tokenized_input_sentence = source_vectorization([input_sentence])
  decoded_sentence = "[start]"
  for i in range(max_decoded_sentence_length):
    tokenized_target_sentence = target_vectorization(
        [decoded_sentence])[:, :-1]
    predictions = transformer(
            [tokenized_input_sentence, tokenized_target_sentence])
    sampled_token_index = np.argmax(predictions[0, i, :])
    sampled_token = spa_index_lookup[sampled_token_index]
    decoded_sentence += " " + sampled_token
    if sampled_token == "[end]":
      break
  return decoded_sentence

test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(20):
  input_sentence = random.choice(test_eng_texts)
  print("-")
  print(input_sentence)
  print(decode_sequence(input_sentence))

-
Get out of my bed.
[start] sal de mi cama [end]
-
Why didn't she help you?
[start] por qué no ella te ayudó [end]
-
No one is to leave.
[start] nadie tiene que ir de vacaciones [end]
-
I have a terrible pain.
[start] tengo un dolor terrible [end]
-
In America, cars drive on the right side of the road.
[start] en los coches coche [UNK] en la derecha de la calle [end]
-
He dislikes me.
[start] Él no me [UNK] [end]
-
How does this work?
[start] qué tal funciona esto [end]
-
Let's visit Tom.
[start] vayamos a tom [end]
-
She handed him a book.
[start] ella le dio un libro [end]
-
Do come in!
[start] entre entrar [end]
-
May I try it on?
[start] puedo hacerlo [end]
-
At first, everything proceeded according to plan.
[start] primero en todo lo que decir plan [end]
-
What time does your plane depart?
[start] a qué hora empieza tu avión [end]
-
I fell asleep with my contacts in.
[start] me quedé dormido con mi [UNK] en el [UNK] [end]
-
Someday your dream will come true.
[start] algún día se 

### 3.2 Inference with Transformer Model

The model appears to be generating reasonable translations, though the `[UNK]` tokens indicate out-of-vocabulary words that could be addressed with a larger vocabulary or subword tokenization.